In [79]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 130

In [80]:
data = 'https://raw.githubusercontent.com/gastonstat/CreditScoring/master/CreditScoring.csv'
!wget $data

--2026-05-29 13:24:07--  https://raw.githubusercontent.com/gastonstat/CreditScoring/master/CreditScoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 182489 (178K) [text/plain]
Saving to: ‘CreditScoring.csv.9’

CreditScoring.csv.9 100%[===================>] 178.21K  --.-KB/s    in 0.02s   

2026-05-29 13:24:07 (7.09 MB/s) - ‘CreditScoring.csv.9’ saved [182489/182489]



In [81]:
df = pd.read_csv(data)

In [82]:
df.head()

,Status,Seniority,Home,Time,Age,Marital,Records,Job,Expenses,Income,Assets,Debt,Amount,Price
0,1,9,1,60,30,2,1,3,73,129,0,0,800,846
1,1,17,1,60,58,3,1,1,48,131,0,0,1000,1658
2,2,10,2,36,46,2,2,3,90,200,3000,0,2000,2985
3,1,0,1,60,24,1,1,1,63,182,2500,0,900,1325
4,1,0,1,36,26,1,1,1,46,107,0,0,310,910


# Data Cleaning and Preparation
- Decoding feature values
- correcting dtypes
- handling null values


In [83]:
df.columns = df.columns.str.lower()
df.head()

,status,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
0,1,9,1,60,30,2,1,3,73,129,0,0,800,846
1,1,17,1,60,58,3,1,1,48,131,0,0,1000,1658
2,2,10,2,36,46,2,2,3,90,200,3000,0,2000,2985
3,1,0,1,60,24,1,1,1,63,182,2500,0,900,1325
4,1,0,1,36,26,1,1,1,46,107,0,0,310,910


In [84]:
df['status'].unique()

array([1, 2, 0])

In [85]:
df.dtypes

,0
status,int64
seniority,int64
home,int64
time,int64
age,int64
marital,int64
records,int64
job,int64
expenses,int64
income,int64


In [86]:
df['status'] = df['status'].astype(int).map({1: 'ok', 2: 'default', 0: 'unk'})



In [87]:
df['marital'].unique()


array([2, 3, 1, 4, 5, 0])

In [88]:
df['marital'] = df['marital'].map({ 1: 'single',
    2: 'married',
    3: 'widow',
    4: 'separated',
    5: 'divorced',
    0: 'unk'})

In [89]:
df['home'] = df['home'].map({ 1: 'rent',
    2: 'owner',
    3: 'private',
    4: 'ignore',
    5: 'parents',
    6: 'other',
    0: 'unk'})

df['records'] = df['records'].map({ 1: 'no',
    2: 'yes',
    0: 'unk'})
df['job'] = df['job'].map({
    1: 'fixed',
    2: 'partime',
    3: 'freelance',
    4: 'others',
    0: 'unk'
})

In [90]:
df[['status', 'seniority', 'home', 'marital', 'job']] = df[['status', 'seniority', 'home', 'marital', 'job']].astype('str')

In [91]:
num_cols = list(df.dtypes[df.dtypes == 'int'].index)
num_cols


['time', 'age', 'expenses', 'income', 'assets', 'debt', 'amount', 'price']

In [92]:
#decoding null values in numerical columns
for col in num_cols:
  df[col] = df[col].replace(df[col].max(), np.nan)

In [93]:
df[num_cols].isnull().sum()

,0
time,1
age,2
expenses,1
income,34
assets,47
debt,18
amount,1
price,1


In [94]:
# Handle null values
for col in num_cols:
  df[col] = df[col].fillna(df[col].median())

In [95]:
df.isnull().sum()

,0
status,0
seniority,0
home,0
time,0
age,0
marital,0
records,0
job,0
expenses,0
income,0


In [96]:
df.head()


,status,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
0,ok,9,rent,60.0,30.0,married,no,freelance,73.0,129.0,0.0,0.0,800.0,846.0
1,ok,17,rent,60.0,58.0,widow,no,fixed,48.0,131.0,0.0,0.0,1000.0,1658.0
2,default,10,owner,36.0,46.0,married,yes,freelance,90.0,200.0,3000.0,0.0,2000.0,2985.0
3,ok,0,rent,60.0,24.0,single,no,fixed,63.0,182.0,2500.0,0.0,900.0,1325.0
4,ok,0,rent,36.0,26.0,single,no,fixed,46.0,107.0,0.0,0.0,310.0,910.0


In [97]:
#Convert the target field into one hot encoding
df['status'] = (df['status'] == 'default').astype('int')

In [98]:
df['status']

,status
0,0
1,0
2,1
3,0
4,0
...,...
4450,1
4451,0
4452,1
4453,0


In [99]:
df['status'].value_counts(normalize=True)

,proportion
status,
0,0.718519
1,0.281481


# Data validation Framework
- Feature one hot encoding DictVectorizer

In [100]:
# Data Validation Framework
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

len(df_train), len(df_test), len(df_val)

(2673, 891, 891)

In [101]:
y_train = df_train['status'].values
y_test = df_test['status'].values
y_val = df_val['status'].values

In [102]:
y_val[:5]

array([0, 0, 1, 1, 0])

In [103]:
del df_train['status']
del df_test['status']
del df_val['status']

In [104]:
df_train.head()

,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
353,14,owner,60.0,30.0,married,no,fixed,60.0,70.0,4000.0,2800.0,600.0,1125.0
990,2,parents,60.0,35.0,married,no,fixed,75.0,104.0,0.0,0.0,1200.0,1677.0
3289,8,rent,36.0,61.0,single,no,fixed,42.0,72.0,0.0,0.0,325.0,450.0
4002,14,owner,60.0,40.0,married,no,fixed,45.0,91.0,0.0,0.0,1100.0,1565.0
1273,2,other,60.0,41.0,separated,no,freelance,35.0,100.0,5000.0,0.0,1200.0,1450.0


In [107]:
train_dicts = df_train.to_dict(orient='records')
test_dicts = df_test.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

In [109]:
from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_test = dv.transform(test_dicts)
X_val = dv.transform(val_dicts)

# XGBOOST library

In [115]:
import xgboost as xgb

dtype('int64')